# MemeX -- sbor datasета predskazatelya ekspertov (put A, nash fork + CUDA)

Trebovaniya k akkauntu Kaggle (inache yacheyki upadut):
1. Akkaunt VERIFICIROVAN PO TELEFONU -- inache Kaggle ne dayot ni GPU (T4), ni internet.
2. Accelerator = GPU T4 x2, Internet = On.
3. Dostup k privatnomu forku: libo sdelat repozitoriy publichnym, libo dobavit sekret GITHUB_PAT (Add-ons -> Secrets) s tokenom, imeyushchim dostup na chtenie ambassadorOfCplusPlus/memex-engine.

Vetka memex (kommit s reestrom arhitektur, delta-set na karte, ExpertStore i damperami MEMEX_HIDDEN_TRACE / MEMEX_EXPERT_TRACE). Model Qwen3-Coder-Next-UD-IQ3_XXS.gguf (28.5 GB) kachaetsya v /kaggle/tmp (tam sotni GB), ne v /kaggle/working (21 GB).


In [ ]:
import os, sys, subprocess, shutil, time
def sh(cmd, env=None, check=True):
    print('+', cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True, env=env)
    if check and r.returncode != 0:
        raise SystemExit('FAILED rc=%d: %s' % (r.returncode, cmd))
    return r.returncode
def out(cmd):
    return subprocess.run(cmd, shell=True, text=True, capture_output=True).stdout
import torch
assert torch.cuda.is_available(), 'NET GPU: vklyuchi accelerator T4 (nuzhna telefonnaya verifikaciya akkaunta)'
print('GPU:', torch.cuda.get_device_name(0), 'x', torch.cuda.device_count())
print('net github:', out('curl -sI https://github.com -m 20 -o /dev/null -w "%{http_code}"'))


In [ ]:
pat = ''
try:
    from kaggle_secrets import UserSecretsClient
    pat = UserSecretsClient().get_secret('GITHUB_PAT').strip()
    print('sekret GITHUB_PAT poluchen')
except Exception as e:
    print('net sekreta GITHUB_PAT (ok tolko esli repozitoriy publichnyy):', e)
ENG = '/kaggle/tmp/engine'
shutil.rmtree(ENG, ignore_errors=True); os.makedirs('/kaggle/tmp', exist_ok=True)
base = 'github.com/ambassadorOfCplusPlus/memex-engine.git'
url = ('https://%s@%s' % (pat, base)) if pat else ('https://%s' % base)
sh('git clone --depth 1 -b memex %s %s' % (url, ENG))
print('HEAD:', out('git -C %s rev-parse HEAD' % ENG).strip())
grep = out('grep -rl MEMEX_HIDDEN_TRACE %s/examples %s/src 2>/dev/null' % (ENG, ENG)).strip()
assert grep, 'V sklonirovannoy vetke NET MEMEX_HIDDEN_TRACE -- ne tot kommit/vetka'
print('damper nayden v:', grep.replace(chr(10), ', '))


In [ ]:
J = os.cpu_count()
def cmake_build(cuda):
    sh('cd %s && rm -rf build' % ENG)
    flag = '-DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75' if cuda else '-DGGML_CUDA=OFF'
    sh('cd %s && cmake -B build %s -DLLAMA_CURL=OFF -DGGML_NATIVE=OFF -DCMAKE_BUILD_TYPE=Release' % (ENG, flag))
    sh('cd %s && cmake --build build --target llama-memex-fwd -j %d' % (ENG, J))
CUDA = True
try:
    cmake_build(True)
except SystemExit as e:
    print('CUDA-sborka upala:', e)
    print('>>> folbek na CPU-only (damp marshrutizacii vsyo ravno snimetsya, medlennee)')
    CUDA = False; cmake_build(False)
cands = out('find %s/build -name llama-memex-fwd -type f' % ENG).strip().splitlines()
assert cands, 'binar llama-memex-fwd ne nayden posle sborki'
EXE = cands[0]; print('EXE:', EXE, '| CUDA:', CUDA)
print(out('%s --help 2>&1 | head -40' % EXE))


In [ ]:
sh('pip -q install huggingface_hub gguf')
from huggingface_hub import hf_hub_download
MODEL = hf_hub_download('unsloth/Qwen3-Coder-Next-GGUF', 'Qwen3-Coder-Next-UD-IQ3_XXS.gguf', local_dir='/kaggle/tmp/model')
print('MODEL:', MODEL, '%.1f GB' % (os.path.getsize(MODEL) / 1e9))


In [ ]:
prompts = {
 'code_py': 'def fibonacci(n):\n    """Return the n-th Fibonacci number."""\n',
 'prose_ru': 'Osen v etom godu prishla rano. Pervye holodnye dozhdi zastuchali po krysham, i gorod',
 'math': 'Teorema. Pust G -- konechnaya gruppa poryadka p^2, gde p prostoe. Dokazhem, chto G abeleva.',
 'sql': '-- Otchyot po prodazham za kvartal po regionam\nSELECT region, SUM(amount) AS total\nFROM sales',
 'dialog': 'User: How do I reverse a singly linked list in place?\nAssistant:',
 'json_cfg': '{\n  "service": "memex-router",\n  "replicas": 3,\n  "resources": {',
}
os.makedirs('/kaggle/tmp/prompts', exist_ok=True); paths = {}
for k, v in prompts.items():
    p = '/kaggle/tmp/prompts/%s.txt' % k; open(p, 'w', encoding='utf-8').write(v); paths[k] = p
print('promtov:', len(paths))


In [ ]:
OUTD = '/kaggle/working/dataset'; os.makedirs(OUTD, exist_ok=True)
TOK = 1500
def run_one(name, pp):
    h = '%s/hidden_%s.bin' % (OUTD, name); r = '%s/route_%s.bin' % (OUTD, name)
    env = dict(os.environ, MEMEX_HIDDEN_TRACE=h, MEMEX_EXPERT_TRACE=r)
    base = '%s -m %s -f %s --tokens %d --gen 2 -t %d --no-repack --no-ref --prefill-chunk 128' % (EXE, MODEL, pp, TOK, os.cpu_count())
    variants = ([base + ' -ngl 99', base] if CUDA else [base])
    for cmd in variants:
        rc = sh(cmd, env=env, check=False)
        if rc == 0 and os.path.exists(h) and os.path.exists(r):
            return h, r
    return None
pairs = []
for name, pp in paths.items():
    res = run_one(name, pp)
    if res:
        pairs.append(res)
        print('OK', name, '%.0f MB hidden, %.0f KB route' % (os.path.getsize(res[0]) / 1e6, os.path.getsize(res[1]) / 1e3))
    else:
        print('PROMAH', name)
print('sobrano par:', len(pairs))
import struct
tot_tok = 0
for h, _ in pairs:
    with open(h, 'rb') as f:
        nt = struct.unpack('<i', f.read(4))[0]; tot_tok += nt
print('vsego tokenov dataseta:', tot_tok)


## Skripty obucheniya (pereneseny iz bench/): route_lab.py, hidden_lab.py, train_r1.py


In [ ]:
%%writefile /kaggle/working/bench/route_lab.py
# -*- coding: utf-8 -*-
"""Лаборатория предсказателей экспертов по следу маршрутизации.

Отвечает на вопрос шага 5 плана: КАКИЕ эксперты держать/подгружать на следующий токен и
следующий слой, если их выбирает не частота, а предсказатель. Всё считается офлайн по следу,
поэтому десяток способов сравнивается за секунды, без пересборки и без замка машины.

Отличия от bench/route_predict.py (база прошлого агента), которые тут добавлены намеренно:
  * ТРИ метрики, а не одна: среднее по всему следу; ХОЛОДНЫЙ СТАРТ (первые N токенов);
    ХУДШИЙ СЛОЙ. Промах случается в конкретном слое и в начале документа, а не «в среднем».
  * Предсказание ВНУТРИ токена: при счёте слоя l уже известны выборы слоёв 0..l-1 этого же
    токена. Способ «межслойный» учится на совместной встречаемости (слой l-k -> слой l) и
    предсказывает слой l на k слоёв вперёд - это ровно то окно, за которое надо успеть
    прочитать эксперта с SSD.
  * Затравка с ЧУЖОГО текста (prior): частоты другого следа как начальные счётчики, чтобы
    померить, спасает ли это холодный старт, и насколько (перенос между текстами 81%).
  * Два режима бюджета: РЕЗИДЕНТНОСТЬ (сколько держать в ОЗУ: 128..410 из 512) и
    ПРЕДЗАГРУЗКА (сколько читать заранее на слой: 10..64).

Честность. Скрипт печатает, чего он НЕ мерит:
  * след снят на нашем промте, а не на собственном продолжении модели;
  * стоимость самого предсказателя не учтена - это потолок способа, а не выигрыш движка;
  * скрытых состояний в следе нет, поэтому предсказатель по скрытому состоянию тут НЕ
    измерен - нужен отдельный дамп (см. --hidden, пока отказ вслух).

Форматы следа:
  новый (MEMEX_EXPERT_TRACE): заголовок int32[4] = n_tokens, n_layer, n_used, n_expert,
      тело [token][layer][slot];
  старый (moe_trace.bin от 30B): блоки [layer, n_used, n_tokens] + ids [token][slot].

Запуск:  python bench/route_lab.py <след> [--prior <другой след>] [--cold 200]
"""
import argparse, io, os, struct, sys, collections
import numpy as np

if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8', errors='replace')

MISS_SSD_MS = 3.616      # измерено bench/ssd_raw_reads.py, блок 1,67 МБ мимо кэша
MISS_HDD_MS = 23.287


# ----------------------------------------------------------------------------- загрузка
def load_trace(path):
    raw = io.open(path, 'rb').read()
    if len(raw) >= 16:
        n_tok, n_layer, n_used, n_exp = struct.unpack('<iiii', raw[:16])
        if (n_tok > 0 and 0 < n_layer <= 256 and 0 < n_used <= 32 and 0 < n_exp <= 4096
                and len(raw) == 16 + 4 * n_tok * n_layer * n_used):
            body = np.frombuffer(raw, dtype='<i4', offset=16)
            return body.reshape(n_tok, n_layer, n_used).copy(), n_exp, 'MEMEX_EXPERT_TRACE'
    # старый формат: блоки по слоям
    # старый формат: блоки [layer, n_used, n_tokens] идут подряд (префилл, затем по токену),
    # блоки одного слоя склеиваются; последний слой в префилле считает только последний токен
    # (эталон обрезает граф), поэтому слои с половиной длины отбрасываются вслух.
    layers = collections.defaultdict(list)
    off = 0
    while off + 12 <= len(raw):
        layer, n_used, n_tokens = struct.unpack('<iii', raw[off:off + 12]); off += 12
        n = n_used * n_tokens
        if off + 4 * n > len(raw):
            break
        ids = np.frombuffer(raw, dtype='<i4', offset=off, count=n).reshape(n_tokens, n_used)
        off += 4 * n
        if layer >= 0:
            layers[layer].append(ids)
    if not layers:
        raise SystemExit('след не разобран ни одним форматом: %s' % path)
    layers = {il: np.concatenate(v) for il, v in layers.items()}
    full = max(v.shape[0] for v in layers.values())
    short = sorted(il for il, v in layers.items() if v.shape[0] < full // 2)
    if short:
        print('  старый формат: слои %s имеют неполный след и ОТБРОШЕНЫ' % short)
        for il in short: del layers[il]
    n_layer = max(layers) + 1
    n_tok = min(v.shape[0] for v in layers.values())
    n_used = next(iter(layers.values())).shape[1]
    arr = np.full((n_tok, n_layer, n_used), -1, dtype=np.int32)
    for il, v in layers.items():
        arr[:, il, :] = v[:n_tok]
    n_exp = int(arr.max()) + 1
    return arr, n_exp, 'moe_trace (блоки по слоям)'


# ----------------------------------------------------------------------------- предсказатели
# Контракт: predictor(il) -> объект с
#   order(ctx)   -> np.ndarray id в порядке убывания приоритета (можно короче n_exp). Бюджет B
#                   означает «первые B из этого порядка», поэтому все бюджеты оцениваются за
#                   один вызов. ctx.same_token[l] - выборы слоёв < il текущего токена (уже
#                   известны), ctx.t - номер токена.
#   observe(cur, ctx) -> учесть фактический выбор слоя il на текущем токене
# Никакого заглядывания вперёд: order вызывается ДО observe того же слоя и токена.

class Ctx:
    __slots__ = ('t', 'same_token')
    def __init__(s): s.t = 0; s.same_token = {}


class Freq:
    "частота по всей истории слоя - база, которую надо побить"
    def __init__(s, n_exp, prior=None, prior_w=0.0):
        s.c = np.zeros(n_exp, dtype=np.float64)
        if prior is not None and prior_w > 0:
            s.c += prior_w * prior / max(prior.sum(), 1.0)
    def order(s, ctx):
        if not s.c.any(): return np.empty(0, dtype=np.int64)
        o = np.argsort(-s.c, kind='stable')
        return o[s.c[o] > 0]
    def observe(s, cur, ctx): np.add.at(s.c, cur, 1.0)


class FreqDecay(Freq):
    "частота с экспоненциальным забыванием: следит за темой, а не за всем документом"
    def __init__(s, n_exp, alpha, **kw):
        super().__init__(n_exp, **kw); s.alpha = alpha
    def observe(s, cur, ctx):
        s.c *= s.alpha
        np.add.at(s.c, cur, 1.0)


class WindowFreq(Freq):
    "объединение последних W токенов, остаток бюджета - частотой"
    def __init__(s, n_exp, W, **kw):
        super().__init__(n_exp, **kw); s.q = collections.deque(maxlen=W)
    def order(s, ctx):
        keep = []
        seen = set()
        for step in reversed(s.q):
            for e in step:
                if e not in seen:
                    seen.add(e); keep.append(e)
        if s.c.any():
            for e in np.argsort(-s.c, kind='stable'):
                if s.c[e] <= 0: break
                if e not in seen: seen.add(e); keep.append(int(e))
        return np.asarray(keep, dtype=np.int64)
    def observe(s, cur, ctx):
        s.q.append(list(cur)); super().observe(cur, ctx)


class LRU:
    def __init__(s, n_exp): s.o = collections.OrderedDict()
    def order(s, ctx): return np.asarray(list(s.o.keys())[::-1], dtype=np.int64)
    def observe(s, cur, ctx):
        for e in cur:
            s.o.pop(int(e), None); s.o[int(e)] = 1


class CrossLayer:
    """Межслойный: слой il предсказывается по выборам слоя il-k ТОГО ЖЕ токена.

    Таблица T[p, e] - сколько раз эксперт p на слое il-k и эксперт e на слое il встретились
    на одном токене. Оценка e = сумма T[p, e] по p из фактического выбора слоя il-k, плюс
    малая доля частоты как разрешение ничьих. Учится онлайн, причинно. Для первых k слоёв
    (нет слоя il-k в этом токене) берётся выбор слоя il на ПРЕДЫДУЩЕМ токене - то, что было
    бы известно движку в этот момент.
    """
    def __init__(s, n_exp, il, k, prior_T=None, prior_w=0.0, tie=0.01):
        s.il = il; s.k = k; s.n = n_exp; s.tie = tie
        s.T = np.zeros((n_exp, n_exp), dtype=np.float32)
        if prior_T is not None and prior_w > 0:
            s.T += prior_w * prior_T / max(prior_T.sum(), 1.0) * n_exp
        s.c = np.zeros(n_exp, dtype=np.float64)
        s.last_src = None      # выбор слоя-источника на этом токене (для observe)
        s.prev_cur = None      # свой выбор на предыдущем токене
    def _src(s, ctx):
        src_l = s.il - s.k
        if src_l >= 0 and src_l in ctx.same_token:
            return ctx.same_token[src_l], True
        return s.prev_cur, False
    def order(s, ctx):
        src, _ = s._src(ctx)
        score = s.c * s.tie
        if src is not None and len(src):
            score = score + s.T[np.asarray(src)].sum(axis=0)
        if not score.any(): return np.empty(0, dtype=np.int64)
        o = np.argsort(-score, kind='stable')
        return o[score[o] > 0]
    def observe(s, cur, ctx):
        src, same = s._src(ctx)
        if src is not None and len(src):
            s.T[np.ix_(np.asarray(src), np.asarray(cur))] += 1.0
        np.add.at(s.c, cur, 1.0)
        s.prev_cur = np.asarray(cur)


# ----------------------------------------------------------------------------- прогон
def run(trace, n_exp, make, budgets, cold_n):
    """Возвращает {B: (среднее, холодный старт, худший слой, доля по слоям)}.
    Один проход по следу на способ: порядок приоритета считается один раз, а попадание при
    бюджете B - это «позиция эксперта в порядке < B», поэтому все бюджеты сразу."""
    n_tok, n_layer, n_used = trace.shape
    budgets = np.asarray(budgets)
    nb = len(budgets)
    preds = [make(il) for il in range(n_layer)]
    hit = np.zeros((n_layer, nb), dtype=np.int64); tot = np.zeros(n_layer, dtype=np.int64)
    hit_cold = np.zeros(nb, dtype=np.int64); tot_cold = 0
    pos = np.empty(n_exp, dtype=np.int64)
    ctx = Ctx()
    for t in range(n_tok):
        ctx.t = t; ctx.same_token = {}
        for il in range(n_layer):
            cur = trace[t, il]
            cur = cur[(cur >= 0) & (cur < n_exp)]
            if len(cur) == 0: continue
            if t > 0:
                o = preds[il].order(ctx)
                pos.fill(n_exp)
                if len(o): pos[o] = np.arange(len(o))
                h = (pos[cur][:, None] < budgets[None, :]).sum(axis=0)
                hit[il] += h; tot[il] += len(cur)
                if t < cold_n:
                    hit_cold += h; tot_cold += len(cur)
            preds[il].observe(cur, ctx)
            ctx.same_token[il] = cur
    out = {}
    for j, B in enumerate(budgets):
        per = np.where(tot > 0, hit[:, j] / np.maximum(tot, 1) * 100.0, np.nan)
        out[int(B)] = (hit[:, j].sum() / max(tot.sum(), 1) * 100.0,
                       hit_cold[j] / max(tot_cold, 1) * 100.0,
                       np.nanmin(per) if np.isfinite(per).any() else float('nan'), per)
    return out


def prior_from(trace, n_exp):
    """Частоты по слоям и таблицы совместной встречаемости (k=1) с чужого текста."""
    n_tok, n_layer, n_used = trace.shape
    freq = np.zeros((n_layer, n_exp)); T1 = np.zeros((n_layer, n_exp, n_exp), dtype=np.float32)
    for il in range(n_layer):
        v = trace[:, il, :]; v = v[(v >= 0) & (v < n_exp)]
        np.add.at(freq[il], v, 1.0)
        if il >= 1:
            for t in range(n_tok):
                a = trace[t, il - 1]; b = trace[t, il]
                a = a[(a >= 0) & (a < n_exp)]; b = b[(b >= 0) & (b < n_exp)]
                T1[il][np.ix_(a, b)] += 1.0
    return freq, T1


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('trace')
    ap.add_argument('--prior', action='append', help='след другого текста для затравки холодного старта (можно несколько раз, частоты суммируются)')
    ap.add_argument('--cold', type=int, default=200, help='длина холодного старта в токенах')
    ap.add_argument('--resident', default='128,192,256,320,384,410',
                    help='бюджеты резидентности (экспертов на слой в ОЗУ)')
    ap.add_argument('--prefetch', default='10,16,24,32,48,64',
                    help='бюджеты предзагрузки (экспертов на слой заранее)')
    ap.add_argument('--ks', default='1,2,4', help='на сколько слоёв вперёд предсказывать')
    ap.add_argument('--hidden', help='дамп скрытых состояний (пока не поддержан)')
    a = ap.parse_args()

    if a.hidden:
        print('ОТКАЗ: предсказатель по скрытому состоянию требует дампа, формат которого ещё '
              'не определён; этот канал НЕ ИЗМЕРЕН')
    trace, n_exp, fmt = load_trace(a.trace)
    n_tok, n_layer, n_used = trace.shape
    per_token = n_layer * n_used
    print('след: %s, формат %s' % (a.trace, fmt))
    print('  %d токенов x %d слоёв x %d мест, из %d экспертов; обращений на токен %d'
          % (n_tok, n_layer, n_used, n_exp, per_token))
    if n_tok * n_used / n_exp < 10:
        print('  ВНИМАНИЕ: %.1f обращений на эксперта на слой - выборка мала, выводы о дрейфе '
              'недействительны (урок HANDOFF_ROUTER §4)' % (n_tok * n_used / n_exp))

    prior_f = prior_T = None
    for pp in (a.prior or []):
        ptr, pexp, pfmt = load_trace(pp)
        if pexp != n_exp or ptr.shape[1] != n_layer:
            print('  prior %s: другая геометрия (%d слоёв, %d экспертов) - НЕ применён' % (pp, ptr.shape[1], pexp))
            continue
        f, T = prior_from(ptr, n_exp)
        prior_f = f if prior_f is None else prior_f + f
        prior_T = T if prior_T is None else prior_T + T
        print('  затравка с чужого текста: %s (%d токенов)' % (pp, ptr.shape[0]))

    resident = [int(x) for x in a.resident.split(',') if int(x) < n_exp]
    prefetch = [int(x) for x in a.prefetch.split(',') if int(x) < n_exp]
    ks = [int(x) for x in a.ks.split(',')]

    ways = [
        ('частота (база)',        lambda il: Freq(n_exp)),
        ('частота, забыв. 0.99',  lambda il: FreqDecay(n_exp, 0.99)),
        ('частота, забыв. 0.995', lambda il: FreqDecay(n_exp, 0.995)),
        ('окно 16 + частота',     lambda il: WindowFreq(n_exp, 16)),
        ('LRU',                   lambda il: LRU(n_exp)),
    ]
    for k in ks:
        ways.append(('межслойный k=%d' % k, (lambda k: lambda il: CrossLayer(n_exp, il, k))(k)))
    if prior_f is not None:
        for w in (50, 200):
            ways.append(('частота + затравка w=%d' % w,
                         (lambda w: lambda il: Freq(n_exp, prior=prior_f[il], prior_w=w))(w)))
        ways.append(('межслойный k=1 + затравка',
                     lambda il: CrossLayer(n_exp, il, 1, prior_T=prior_T[il], prior_w=200)))

    def table(title, budgets):
        print('\n%s' % title)
        print('  %-28s' % 'способ' + ''.join('%22d' % B for B in budgets))
        print('  %-28s' % '' + ''.join('%22s' % 'сред/холод/худш.слой' for B in budgets))
        rows = []
        for name, mk in ways:
            r = run(trace, n_exp, mk, budgets, a.cold)
            rows.append((name, r))
            line = '  %-28s' % name
            for B in budgets:
                m, c, w, _ = r[B]
                line += '   %6.2f/%6.2f/%6.2f' % (m, c, w)
            print(line); sys.stdout.flush()
        return rows

    rows_r = table('РЕЗИДЕНТНОСТЬ: доля попаданий на следующий токен, %% (бюджет = экспертов '
                   'на слой в ОЗУ; холодный старт = первые %d токенов)' % a.cold, resident)
    base = dict(rows_r)['частота (база)']
    print('\nЦЕНА ПРОМАХОВ БАЗЫ (частота) в мс на токен, среднее / холодный старт:')
    for B in resident:
        m, c, w, _ = base[B]
        print('  бюджет %3d/%d: SSD %6.1f / %6.1f мс   HDD %6.1f / %6.1f мс'
              % (B, n_exp, (100 - m) / 100 * per_token * MISS_SSD_MS,
                 (100 - c) / 100 * per_token * MISS_SSD_MS,
                 (100 - m) / 100 * per_token * MISS_HDD_MS,
                 (100 - c) / 100 * per_token * MISS_HDD_MS))
    print('\nЛУЧШИЙ СПОСОБ ПРОТИВ ЧАСТОТЫ, пункты покрытия -> мс на токен с SSD (среднее; холодный старт):')
    for B in resident:
        bm = max(rows_r, key=lambda r: r[1][B][0]); bc = max(rows_r, key=lambda r: r[1][B][1])
        dm = bm[1][B][0] - base[B][0]; dc = bc[1][B][1] - base[B][1]
        print('  бюджет %3d: %-26s %+6.2f п. -> %6.1f мс ; холод: %-26s %+6.2f п. -> %6.1f мс'
              % (B, bm[0], dm, dm / 100 * per_token * MISS_SSD_MS,
                 bc[0], dc, dc / 100 * per_token * MISS_SSD_MS))

    table('ПРЕДЗАГРУЗКА: доля попаданий при чтении B экспертов на слой заранее, %% '
          '(это точность самого предсказания; top-%d = идеал)' % n_used, prefetch)

    print('\nЧЕГО ЗДЕСЬ НЕ ИЗМЕРЕНО:')
    print('  - след снят на нашем промте, а не на собственном продолжении модели')
    print('  - цена самого предсказателя не учтена: потолок способа, а не выигрыш движка')
    print('  - скрытые состояния: канала нет, предсказатель по ним НЕ измерен')
    print('  - межслойный способ предсказывает слой il по слою il-k того же токена; для первых '
          'k слоёв - по своему выбору на предыдущем токене')
    if prior_f is None:
        print('  - затравка с чужого текста: не задан --prior, холодный старт с затравкой НЕ измерен')


if __name__ == '__main__':
    main()



In [ ]:
%%writefile /kaggle/working/bench/hidden_lab.py
# -*- coding: utf-8 -*-
"""Предсказатель экспертов по СКРЫТОМУ СОСТОЯНИЮ - офлайн-оценка по дампу.

Идея пользователя: микромодель, которой на вход идёт скрытое состояние, а на выходе -
вероятности экспертов на ближайшую перспективу. Здесь она проверяется без движка и без
карты, по дампу скрытых состояний и следу маршрутизации.

Что именно предсказывается. В момент, когда слой l токена t уже посчитал вход своего
маршрутизатора x_l (это выход ffn_norm), нужно назвать эксперты слоёв l+1..l+k того же
токена - чтобы их успели прочитать. Три способа, по нарастанию стоимости:

  R0  «свой маршрутизатор»: router_{l+k} · x_l. Ноль новых весов: маршрутизаторы уже лежат на
      карте, это один матвектор [2048x512]. Работает, если остаточный поток меняется от слоя к
      слою медленно. Проверка k=0 обязана дать ~100% - иначе дамп и след не согласованы.
  R1  «подправленный маршрутизатор»: линейная поправка поверх R0, обученная гребневой
      регрессией на первой половине документа, проверенная на второй.
  R2  «свой линейный слой»: W_{l,k} x_l -> логиты слоя l+k, обучен гребневой регрессией на
      точные логиты слоя l+k (они известны: router_{l+k} · x_{l+k}). Полный ранг 2048x512 -
      4 МиБ f32 на пару (l,k), в движок пойдёт только в низком ранге; здесь это ПОТОЛОК.

Метрики те же, что в route_lab.py: среднее, холодный старт (первые N токенов - здесь у
скрытого состояния нет истории, так что холодный старт должен быть равен среднему), худший
слой. Бюджеты предзагрузки: 10..64 экспертов на слой.

Формат дампа (MEMEX_HIDDEN_TRACE, пишет движок): заголовок int32[4] = n_tokens, n_layer,
n_embd, dtype (1 = f16, 0 = f32); тело [token][layer][n_embd] - выход ffn_norm, то есть вход
маршрутизатора. Порядок токенов тот же, что в следе MEMEX_EXPERT_TRACE.

Честность: скрипт печатает, что НЕ измерено. Для проверки водопровода есть --selftest:
случайные маршрутизаторы и медленно дрейфующий поток, k=0 обязан дать 100%.

Запуск: python bench/hidden_lab.py <дамп> <след> <модель.gguf> [--ks 1,2,4] [--budgets 10,16,24,32,48,64]
"""
import argparse, io, struct, sys, os
import numpy as np

if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8', errors='replace')

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from route_lab import load_trace  # noqa: E402


def load_hidden(path):
    raw = io.open(path, 'rb').read()
    n_tok, n_layer, n_embd, dt = struct.unpack('<iiii', raw[:16])
    dtype = np.float16 if dt == 1 else np.float32
    body = np.frombuffer(raw, dtype=dtype, offset=16)
    need = n_tok * n_layer * n_embd
    if body.size != need:
        raise SystemExit('дамп: тело %d элементов, заголовок обещает %d - ДАМП НЕДЕЙСТВИТЕЛЕН'
                         % (body.size, need))
    return body.reshape(n_tok, n_layer, n_embd).astype(np.float32)


def load_routers(gguf_path, n_layer):
    sys.path.insert(0, r'D:\MemeX\src\ik_llama.cpp\gguf-py')
    from gguf import GGUFReader
    r = GGUFReader(gguf_path)
    by = {t.name: t for t in r.tensors}
    W = []
    for il in range(n_layer):
        t = by['blk.%d.ffn_gate_inp.weight' % il]
        if t.tensor_type.name != 'F32':
            raise SystemExit('маршрутизатор слоя %d не F32 (%s) - деквантизация не реализована'
                             % (il, t.tensor_type.name))
        # gguf хранит [n_expert, n_embd] строками (ne0 = n_embd быстрее всего)
        W.append(np.asarray(t.data, dtype=np.float32).reshape(-1, int(t.shape[0])))
    return np.stack(W)   # [n_layer, n_expert, n_embd]


def topk_sets(logits, k):
    return np.argpartition(-logits, k - 1, axis=-1)[..., :k]


def hits_all_budgets(score, actual, budgets):
    """score [T, E], actual [T, k] -> попадания на каждый бюджет [nb]"""
    order = np.argsort(-score, axis=-1, kind='stable')
    pos = np.empty_like(order)
    T, E = score.shape
    pos[np.arange(T)[:, None], order] = np.arange(E)[None, :]
    p = np.take_along_axis(pos, actual, axis=-1)          # [T, k]
    return np.array([(p < B).sum() for B in budgets])


def ridge(X, Y, lam):
    """W = argmin ||XW - Y||^2 + lam||W||^2, X [T, d] (со столбцом 1), Y [T, m]"""
    A = X.T @ X + lam * np.eye(X.shape[1], dtype=np.float64)
    return np.linalg.solve(A, X.T @ Y)


def evaluate(H, trace, W, ks, budgets, cold_n, lam=10.0):
    n_tok, n_layer, n_embd = H.shape
    n_used = trace.shape[2]
    n_exp = W.shape[1]
    half = n_tok // 2
    ways = ['R0 свой маршрутизатор', 'R1 подправленный', 'R2 свой линейный (потолок)']
    res = {}
    for k in [0] + ks:
        acc = {w: {'hit': np.zeros(len(budgets)), 'tot': 0, 'cold': np.zeros(len(budgets)),
                   'coldtot': 0, 'per': []} for w in ways}
        for l in range(n_layer - k):
            tgt = l + k
            actual = trace[:, tgt, :]
            ok = (actual >= 0).all(axis=1)
            X = H[:, l, :]                               # [T, d]
            logit0 = X @ W[tgt].T                        # R0: [T, E]
            true_logit = H[:, tgt, :] @ W[tgt].T         # точные логиты цели
            # обучение на первой половине, проверка на второй
            tr = np.arange(n_tok) < half
            te = ~tr & ok
            Xb = np.concatenate([X, np.ones((n_tok, 1), np.float32)], axis=1).astype(np.float64)
            L0b = np.concatenate([logit0, np.ones((n_tok, 1), np.float32)], axis=1).astype(np.float64)
            scores = {ways[0]: logit0}
            if k > 0:
                W1 = ridge(L0b[tr & ok], true_logit[tr & ok].astype(np.float64), lam)
                scores[ways[1]] = (L0b @ W1).astype(np.float32)
                W2 = ridge(Xb[tr & ok], true_logit[tr & ok].astype(np.float64), lam)
                scores[ways[2]] = (Xb @ W2).astype(np.float32)
            for w, sc in scores.items():
                # k=0 и R0 меряются на всём следе (нет обучения); обученные - на второй половине
                mask = ok if w == ways[0] else te
                h = hits_all_budgets(sc[mask], actual[mask], budgets)
                acc[w]['hit'] += h; acc[w]['tot'] += mask.sum() * n_used
                acc[w]['per'].append(h / max(mask.sum() * n_used, 1) * 100)
                cm = mask & (np.arange(n_tok) < cold_n)
                if cm.any():
                    acc[w]['cold'] += hits_all_budgets(sc[cm], actual[cm], budgets)
                    acc[w]['coldtot'] += cm.sum() * n_used
        for w in ways:
            if acc[w]['tot'] == 0: continue
            per = np.array(acc[w]['per'])                 # [layers, nb]
            res[(k, w)] = (acc[w]['hit'] / acc[w]['tot'] * 100,
                           acc[w]['cold'] / max(acc[w]['coldtot'], 1) * 100,
                           per.min(axis=0))
    return res, ways


def print_table(res, ways, ks, budgets, n_used):
    print('\nПРЕДЗАГРУЗКА ПО СКРЫТОМУ СОСТОЯНИЮ: доля попаданий, %% (сред/холод/худш.слой); '
          'top-%d = идеал' % n_used)
    print('  %-30s' % 'k / способ' + ''.join('%22d' % B for B in budgets))
    for k in [0] + ks:
        for w in ways:
            if (k, w) not in res: continue
            m, c, worst = res[(k, w)]
            line = '  k=%d %-26s' % (k, w)
            for j in range(len(budgets)):
                # обученные способы проверяются на второй половине, где холодного старта нет
                cold = '%6.2f' % c[j] if w == ways[0] else '   н/д'
                line += '   %6.2f/%s/%6.2f' % (m[j], cold, worst[j])
            print(line)
    if (0, ways[0]) in res:
        m0 = res[(0, ways[0])][0][0]
        if m0 < 99.0:
            print('  <<< ПРОВЕРКА k=0 НЕ ПРОЙДЕНА: %.2f%% вместо ~100 - дамп и след не согласованы '
                  '(порядок токенов, слой, нормировка). Остальные строки НЕДЕЙСТВИТЕЛЬНЫ.' % m0)
        else:
            print('  проверка k=0: %.2f%% - дамп и след согласованы' % m0)


def selftest(ks, budgets):
    rng = np.random.default_rng(1)
    T, L, D, E, K = 400, 12, 64, 96, 8
    W = rng.normal(size=(L, E, D)).astype(np.float32)
    H = np.empty((T, L, D), np.float32)
    x = rng.normal(size=(T, D)).astype(np.float32)
    for l in range(L):
        x = x + 0.15 * rng.normal(size=(T, D)).astype(np.float32)
        H[:, l, :] = x
    trace = np.empty((T, L, K), np.int32)
    for l in range(L):
        trace[:, l, :] = topk_sets(H[:, l, :] @ W[l].T, K)
    res, ways = evaluate(H, trace, W, ks, [b for b in budgets if b < E], 100)
    print_table(res, ways, ks, [b for b in budgets if b < E], K)
    print('\nэто САМОПРОВЕРКА на случайных данных: цифры ничего не говорят о модели')


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('hidden', nargs='?')
    ap.add_argument('trace', nargs='?')
    ap.add_argument('gguf', nargs='?')
    ap.add_argument('--ks', default='1,2,4')
    ap.add_argument('--budgets', default='10,16,24,32,48,64')
    ap.add_argument('--cold', type=int, default=200)
    ap.add_argument('--selftest', action='store_true')
    a = ap.parse_args()
    ks = [int(x) for x in a.ks.split(',')]
    budgets = [int(x) for x in a.budgets.split(',')]
    if a.selftest:
        return selftest(ks, budgets)
    if not (a.hidden and a.trace and a.gguf):
        raise SystemExit('нужны дамп, след и gguf (или --selftest)')
    trace, n_exp, fmt = load_trace(a.trace)
    H = load_hidden(a.hidden)
    n_tok, n_layer, n_embd = H.shape
    print('дамп: %d токенов x %d слоёв x %d; след: %s, %d экспертов' % (n_tok, n_layer, n_embd, fmt, n_exp))
    if trace.shape[0] != n_tok or trace.shape[1] != n_layer:
        n = min(trace.shape[0], n_tok)
        print('  длины не совпадают (след %d, дамп %d) - берётся общий префикс %d токенов; '
              'если они с разных прогонов, результат НЕДЕЙСТВИТЕЛЕН' % (trace.shape[0], n_tok, n))
        trace = trace[:n]; H = H[:n]
    W = load_routers(a.gguf, n_layer)
    res, ways = evaluate(H, trace, W, ks, budgets, a.cold)
    print_table(res, ways, ks, budgets, trace.shape[2])
    print('\nЧЕГО НЕ ИЗМЕРЕНО:')
    print('  - стоимость предсказателя в движке (R0 = один матвектор [2048x512] на пару слоёв)')
    print('  - R1/R2 обучены на первой половине ЭТОГО документа; перенос на другой текст не измерен')
    print('  - дамп снят на нашем промте, а не на собственном продолжении модели')


if __name__ == '__main__':
    main()



In [ ]:
%%writefile /kaggle/working/bench/train_r1.py
# -*- coding: utf-8 -*-
"""Обучение и экспорт поправок R1 для предзагрузки экспертов (шаг 5).

R1: логиты слоя l+k ≈ [router_{l+k} · x_l ; 1] · Wc_{l+k}, где Wc — матрица [513×512], обученная
гребневой регрессией на точные логиты слоя l+k. По результатам hidden_lab.py на Coder Next при
бюджете 16 и k=1 даёт 96,8 % (худший слой 90,4 %), при k=4 — 90,2 % (83,7 %).

Экспорт для движка (файл читается в C++ одним fread):
  заголовок int32[4] = {n_layer, n_in (=n_expert+1), n_out (=n_expert), k}
  тело f16 [n_layer][n_in][n_out]; для целевых слоёв tgt < k матрица нулевая с единичной
  диагональю на первых n_out строках (то есть тождество: R1 = R0), чтобы движок не ветвился.

Честность: обучение на первой половине документа; печатается контроль на второй половине для
бюджетов 10/16/32 (тот же метод, что в hidden_lab.py), и предупреждение, что перенос на другие
тексты не измерен — для боевого файла надо обучать на нескольких дампах (--hidden/--trace можно
повторять, они конкатенируются).

Запуск: python bench/train_r1.py --hidden h.bin --trace t.bin --gguf model.gguf --k 4 --out r1_k4.bin
"""
import argparse, io, struct, sys, os
import numpy as np

if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8', errors='replace')
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from route_lab import load_trace  # noqa: E402
from hidden_lab import load_hidden, load_routers, ridge, hits_all_budgets  # noqa: E402


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--hidden', action='append', required=True)
    ap.add_argument('--trace', action='append', required=True)
    ap.add_argument('--gguf', required=True)
    ap.add_argument('--k', type=int, default=4)
    ap.add_argument('--lam', type=float, default=10.0)
    ap.add_argument('--out', required=True)
    ap.add_argument('--budgets', default='10,16,32')
    a = ap.parse_args()
    if len(a.hidden) != len(a.trace):
        raise SystemExit('число дампов и следов должно совпадать (пары)')
    Hs, Ts = [], []
    for h, t in zip(a.hidden, a.trace):
        H = load_hidden(h); T, n_exp, _ = load_trace(t)
        n = min(H.shape[0], T.shape[0]); Hs.append(H[:n]); Ts.append(T[:n])
        print('пара: %s + %s, %d токенов' % (h, t, n))
    H = np.concatenate(Hs); trace = np.concatenate(Ts)
    n_tok, n_layer, n_embd = H.shape
    W = load_routers(a.gguf, n_layer)
    n_out = W.shape[1]; n_in = n_out + 1
    k = a.k
    budgets = [int(x) for x in a.budgets.split(',')]
    half = n_tok // 2
    tr = np.arange(n_tok) < half; te = ~tr
    corr = np.zeros((n_layer, n_in, n_out), dtype=np.float32)
    for tgt in range(n_layer):
        if tgt < k:
            corr[tgt, :n_out, :] = np.eye(n_out, dtype=np.float32)
            continue
        l = tgt - k
        logit0 = H[:, l, :] @ W[tgt].T
        true_logit = H[:, tgt, :] @ W[tgt].T
        L0b = np.concatenate([logit0, np.ones((n_tok, 1), np.float32)], axis=1).astype(np.float64)
        Wc = ridge(L0b[tr], true_logit[tr].astype(np.float64), a.lam)
        corr[tgt] = Wc.astype(np.float32)
    # контроль на второй половине (f16-округлённые веса, как их увидит движок)
    corr16 = corr.astype(np.float16).astype(np.float32)
    hit = np.zeros(len(budgets)); tot = 0; worst = np.full(len(budgets), 100.0)
    for tgt in range(k, n_layer):
        l = tgt - k
        L0b = np.concatenate([H[te, l, :] @ W[tgt].T, np.ones((te.sum(), 1), np.float32)], axis=1)
        sc = L0b @ corr16[tgt]
        actual = trace[te, tgt, :]
        h = hits_all_budgets(sc, actual, budgets)
        hit += h; tot += actual.size
        worst = np.minimum(worst, h / actual.size * 100)
    print('контроль R1 (f16) на второй половине, k=%d: ' % k +
          ', '.join('B=%d %.2f%% (худш. слой %.2f%%)' % (B, hit[i] / tot * 100, worst[i]) for i, B in enumerate(budgets)))
    with io.open(a.out, 'wb') as f:
        f.write(struct.pack('<iiii', n_layer, n_in, n_out, k))
        f.write(corr.astype('<f2').tobytes())
    sz = os.path.getsize(a.out)
    print('записано %s: %d байт = 16 + %d*%d*%d*2' % (a.out, sz, n_layer, n_in, n_out))
    print('НЕ ИЗМЕРЕНО: перенос на другие тексты (обучено на %d документе(ах)); стоимость в движке' % len(Hs))


if __name__ == '__main__':
    main()



In [ ]:
assert len(pairs) >= 2, 'nuzhno >=2 par dlya perenosa mezhdu tekstami'
hargs = ' '.join('--hidden %s --trace %s' % (h, r) for h, r in pairs)
sh('cd /kaggle/working/bench && python train_r1.py %s --gguf %s --k 4 --out /kaggle/working/r1_corr_multi.bin' % (hargs, MODEL))
print('r1_corr_multi.bin:', os.path.getsize('/kaggle/working/r1_corr_multi.bin'), 'bayt')


In [ ]:
sys.path.insert(0, '/kaggle/working/bench')
import numpy as np
from hidden_lab import load_hidden, load_routers, ridge, hits_all_budgets
from route_lab import load_trace
Hs = [load_hidden(h) for h, _ in pairs]
Ts = [load_trace(r)[0] for _, r in pairs]
names = list(paths.keys())[:len(pairs)]
n_layer = Hs[0].shape[1]
W = load_routers(MODEL, n_layer); n_out = W.shape[1]
K = 4; LAM = 10.0; B = 16
def fit_eval(train_idx, test_idx):
    Htr = np.concatenate([Hs[i] for i in train_idx]); Ttr = np.concatenate([Ts[i] for i in train_idx])
    Hte = Hs[test_idx]; Tte = Ts[test_idx]
    hit = 0; tot = 0
    for tgt in range(K, n_layer):
        l = tgt - K
        L0 = np.concatenate([Htr[:, l, :] @ W[tgt].T, np.ones((Htr.shape[0], 1), np.float32)], 1).astype(np.float64)
        true = (Htr[:, tgt, :] @ W[tgt].T).astype(np.float64)
        Wc = ridge(L0, true, LAM).astype(np.float16).astype(np.float32)
        L0e = np.concatenate([Hte[:, l, :] @ W[tgt].T, np.ones((Hte.shape[0], 1), np.float32)], 1)
        sc = L0e @ Wc; actual = Tte[:, tgt, :]
        h = hits_all_budgets(sc, actual, [B]); hit += h[0]; tot += actual.size
    return hit / tot * 100
n = len(pairs)
print('perenos R1 mezhdu tekstami, B=%d k=%d:' % (B, K))
print('%-10s %12s %12s %8s' % ('tekst', 'na_others%', 'na_sebe%', 'razryv'))
gaps = []
for i in range(n):
    tr = [j for j in range(n) if j != i]
    multi = fit_eval(tr, i); solo = fit_eval([i], i)
    gaps.append(solo - multi)
    print('%-10s %12.2f %12.2f %8.2f' % (names[i], multi, solo, solo - multi))
print('sredniy razryv (pereobuchenie na odnom tekste):', round(sum(gaps) / len(gaps), 2), '%')


## Zabrat rezultat
r1_corr_multi.bin lezhit v /kaggle/working/. Skachivanie:
kaggle kernels output ratmirgfxc/memex-dataset-r1 -p D:\\MemeX\\results

### Folbek (put B), esli nash fork ne sobrat s CUDA
CPU-only sborka forka vsyo ravno snimaet MEMEX_EXPERT_TRACE i MEMEX_HIDDEN_TRACE (medlennee). Otdelnyy upstream-put (llama.cpp master podderzhivaet qwen3next) ne nuzhen, poka fork dostupen.
